To create a baseline estimate of what would be useful for the model to achieve, for the elections 2005, 2010,2015, 2017 and 2019, I want to find the following: 

- The accuracy when predicting the constituency winner by assuming no change from previous election.
- The accuracy when predicting the consituency winner using a random forest model
- The accuracy among seats that chance when using this model.

I have selected a random forest model here because I am less interested about interpretability at this moment and purely want to see how we can expect a model to perform on classification. 

In [3]:
import pandas as pd

train_data = pd.read_csv('../Data/test train/TRAIN.csv')

need_cols = ['election', 'country/region','previous_majority_proportion',
       'previous_winner','Conservative', 'Labour', 'LD', 'incumbent','winner']

num_columns = ['previous_majority_proportion', 'Conservative', 'Labour', 'LD']
cat_columns = ['country/region', 'previous_winner', 'incumbent']

train_data1 = train_data[need_cols]

train_data1 = train_data[train_data['winner'] != 'oth'].copy()

In [ ]:
### Output variable types for each column in train_data
pd.DataFrame({
    'column': train_data.columns,
    'dtype': train_data1.dtypes.astype(str).values
})

,column,dtype
0,election,int64
1,country/region,str
2,previous_majority_proportion,float64
3,previous_winner,str
4,Conservative,float64
5,Labour,float64
6,LD,float64
7,incumbent,str
8,winner,str


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

election_years = [2005, 2010, 2015, 2017, 2019]
feature_columns = num_columns + cat_columns

results = []

for election_year in election_years:
    test = train_data1[train_data1['election'] == election_year]
    previous_elections = train_data1[train_data1['election'] < election_year]

    changed_seats = test['previous_winner'] != test['winner']

    previous_winner_predictions = test['previous_winner']
    previous_winner_accuracy = accuracy_score(test['winner'], previous_winner_predictions)
    
    X_train = pd.get_dummies(previous_elections[feature_columns])
    y_train = previous_elections['winner']
    X_test = pd.get_dummies(test[feature_columns]).reindex(columns=X_train.columns, fill_value=0)

    random_forest = RandomForestClassifier(random_state=22)
    random_forest.fit(X_train, y_train)
    random_forest_predictions = random_forest.predict(X_test)

    random_forest_accuracy = accuracy_score(test['winner'], random_forest_predictions)
    random_forest_changed_accuracy = accuracy_score(
        test.loc[changed_seats, 'winner'],
        random_forest_predictions[changed_seats]
    )

    results.append({
        'election': election_year,
        'previous_winner_accuracy': previous_winner_accuracy,
        'random_forest_accuracy': random_forest_accuracy,
        'random_forest_changed_seats_accuracy': random_forest_changed_accuracy,
        'changed_seats': changed_seats.sum()
    })

results = pd.DataFrame(results)
accuracy_columns = [
    'previous_winner_accuracy',
    'random_forest_accuracy',
    'random_forest_changed_seats_accuracy'
]

results[accuracy_columns] = results[accuracy_columns].round(3)
results

,election,previous_winner_accuracy,random_forest_accuracy,random_forest_changed_seats_accuracy,changed_seats
0,2005,0.912,0.904,0.036,55
1,2010,0.825,0.833,0.536,110
2,2015,0.830,0.748,0.028,107
3,2017,0.894,0.892,0.060,67
4,2019,0.881,0.881,0.133,75


Here it is not interesting how the random forest for 2010 has such strong accuracy for the changed seats. It is likely because at this point there had only been two unique entries for the polling figures. As seen in the polling line chart in 01_baseline models.IPYNB, the polling for conservatives went up as well as their seats and the polling for labour went down as well as their seats. The model therefore presumably leanrs this. Hence, the 2019 prediciton is probably most illustrative of a fair accuracy and we can see that it barely differs from predicting the same as last election.

Since we cannot imporve the data by adding more elections going backwards, I am attempting to imporve the model by feature engineering. Specifically, by including introducing a new projected share for each party in each constituency. I created this in 4_projected_polling.sql. The feature is defined as the following:

$$
\begin{aligned}
\text{party X projected share} ={}& \text{party X share at last election} \\
&+ \text{party X national polling} \\
&- \text{party X share of constituencies at last election}
\end{aligned}
$$

In [7]:
need_cols2 = need_cols = ['election','winner', 'country/region','previous_majority_proportion',
       'previous_winner','Conservative', 'Labour', 'LD', 'incumbent',
       'previous_con_share', 'previous_lib_share', 'previous_lab_share',
       'previous_natSW_share', 'projected_con_share', 'projected_lib_share', 'projected_lab_share'
]

train_data2 = train_data[need_cols2]
train_data2 = train_data2[train_data2['winner'] != 'oth'].copy()

In [9]:

election_years = [2005, 2010, 2015, 2017, 2019]
feature_columns = need_cols2[2:]  # Exclude 'election', and 'winner'

results = []

for election_year in election_years:
    test = train_data2[train_data2['election'] == election_year]
    previous_elections = train_data2[train_data2['election'] < election_year]

    changed_seats = test['previous_winner'] != test['winner']

    previous_winner_predictions = test['previous_winner']
    previous_winner_accuracy = accuracy_score(test['winner'], previous_winner_predictions)
    
    X_train = pd.get_dummies(previous_elections[feature_columns])
    y_train = previous_elections['winner']
    X_test = pd.get_dummies(test[feature_columns]).reindex(columns=X_train.columns, fill_value=0)

    random_forest = RandomForestClassifier(random_state=22)
    random_forest.fit(X_train, y_train)
    random_forest_predictions = random_forest.predict(X_test)

    random_forest_accuracy = accuracy_score(test['winner'], random_forest_predictions)
    random_forest_changed_accuracy = accuracy_score(
        test.loc[changed_seats, 'winner'],
        random_forest_predictions[changed_seats]
    )

    results.append({
        'election': election_year,
        'previous_winner_accuracy': previous_winner_accuracy,
        'random_forest_accuracy': random_forest_accuracy,
        'random_forest_changed_seats_accuracy': random_forest_changed_accuracy,
        'changed_seats': changed_seats.sum()
    })

results = pd.DataFrame(results)
accuracy_columns = [
    'previous_winner_accuracy',
    'random_forest_accuracy',
    'random_forest_changed_seats_accuracy'
]

results[accuracy_columns] = results[accuracy_columns].round(3)
results

,election,previous_winner_accuracy,random_forest_accuracy,random_forest_changed_seats_accuracy,changed_seats
0,2005,0.912,0.913,0.018,55
1,2010,0.825,0.806,0.127,110
2,2015,0.830,0.826,0.000,107
3,2017,0.894,0.895,0.075,67
4,2019,0.881,0.844,0.093,75
